In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score,classification_report
import numpy as np

In [2]:
df = pd.read_csv('/content/insurance.csv')

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
70,69,99.9,1.65,0.57,False,Chandigarh,retired,High
9,58,74.4,1.73,43.07,False,Pune,business_owner,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
97,52,60.8,1.80,44.86,False,Hyderabad,freelancer,Low
24,50,54.2,1.66,18.60,False,Mysore,private_job,Medium


In [4]:
df_feat=df.copy()

In [5]:
# FEATURE 1 -> BMI
df_feat["bmi"] = df_feat["weight"]/(df_feat["height"]**2)

In [6]:
# FEATURE 2 -> AGE GROUP
def age_group(age):
  if age<25:
    return "young"
  elif age<45:
    return "adult"
  elif age<60:
    return "middle_age"
  return "senior"

In [7]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [9]:
# FEATURE 3 -> LIFESTYLE RISK
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"]>30:
    return "high"
  elif row["smoker"] and row["bmi"]>27:
    return "medium"
  else:
    return "low"

In [10]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk,axis=1)

In [11]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:
# FEATURE 4 -> City tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [15]:
df_feat['city_tier'] = df_feat["city"].apply(city_tier)

In [17]:
df_feat.drop(columns=["age","weight","height","smoker","city"])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)


,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
77,0.61,retired,37.818734,senior,high,1,High
49,2.29,student,42.701490,young,low,3,Medium
29,50.00,business_owner,42.749310,senior,high,2,High
51,28.95,private_job,38.827923,middle_age,high,2,High
34,0.68,retired,32.914286,senior,high,3,High


In [23]:
# SELECTING FEATURES AND TARGET
X = df_feat[["bmi","age_group","lifestyle_risk","city_tier","income_lpa","occupation"]]
Y = df_feat["insurance_premium_category"]

In [20]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,low,2,2.92000,retired
1,30.189017,adult,low,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,young,high,1,3.34000,student
4,24.296875,senior,low,2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,low,1,34.01000,private_job
97,18.765432,middle_age,low,1,44.86000,freelancer
98,30.521676,adult,low,1,28.30000,business_owner


In [24]:
Y

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High
...,...
95,Low
96,Low
97,Low
98,Low


In [25]:
# DEFINING CATEGORIACL AND NUMERICAL FEATURES
categorical_features = ["age_group","lifestyle_risk","occupation","city_tier"]
numerical_features = ["bmi","income_lpa"]

In [26]:
# CREATING COLUMN TRANSFORMER FOR OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(),categorical_features),
        ("num","passthrough",numerical_features)
    ]
)

In [27]:
# CREATING PIPELINE WITH PROCESSING AND RANDOM FOREST CLASSIFIER
pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(random_state=42))
])

In [28]:
# SPLIT DATA , TRAIN MODEL
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=1)
pipeline.fit(X_train,Y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [29]:
# PREDICT AND EVALUATE
Y_pred = pipeline.predict(X_test)
accuracy_score(Y_test,Y_pred)

0.75

In [30]:
import pickle
# SAVING THE TRAINED PIPELINE USING PICKLE
pickle_model_path = "model.pkl"
with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline,f)